Una vez ejecutadas las funciones de lectura y transformaciones comunes se realizaron las siguientes operaciones sobre los DFs que se emplean como dimensiones o descripción de los códigos encontrados en tabla prestamos.

En este notebook se presentan las siguientes tablas:

- Población: Tabla donde se muestran valores de población censada por barrio y distrito desde 2018 a 2026 cantidades totales y por género. Tabla final, población total por distrito y año.

- Directorio bibliotecas: Información general de las bibliotecas de la red.

- código lector destino (prcoc1): Tabla en la que se describen los códigos de catalogación del público objetivo al que está destinado el volumen. (Adulto, joven, infantil...).

- prprsu: Tabla de códigos alfanuméricos que describen bibliotecas de la red pública de Madrid. Se extrae nombre del distrito para unir con tabla directorio bibliotecas y conseguir código distrito. Tabla final con el código de la biblioteca, nombre, código distrito, nombre distrito y código del directorio. De los 64 códigos de bibliotecas, 12 no pertenecen a bibliotecas reales, son códigos administrativos, los mismos que se retiran líneas tabla prestamos.

- prcolp: Tabla que describe los códigos alfabéticos que identifican el tipo de lector que ha realizado el préstamo. Categorización amplia, engloba tipos como lector, biblioteca, usuario-ebook... En el análisis solo uso de identificadores individuales (lector, profesor, investigador...)

- prcocp: Tabla que describe los códigos alfabéticos del tipo de ejemplar prestado. Se filtrará la tabla por los préstamos que sean prestable libro (prl).

- prlebi: Tabla que describe los códigos alfabéticos que identifican la red de bibliotecas donde el usuario realizó el carnet. Puede ser BPM (biblioteca pública de Madrid) u otros.

- prlesu: Tabla que describe los códigos alfabéticos que identifican la biblioteca pública donde el usuario realizó el carnet. Mismos códigos que prprsu.

- prcocs: Tabla que describe los códigos alfabéticos del tipo de soporte del ejemplar prestado. La tabla préstamos se lee filtrando por aquellos registros que sean libros (lib), no hay más tipos dentro del objeto de análisis.




# Acciones iniciales

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path# Gestionar rutas archivos
import json # Para crear y tratar archivo json
import pandas as pd
from IPython.display import display


pd.options.mode.copy_on_write = True
pd.set_option("io.parquet.engine", "pyarrow")
pd.set_option("mode.string_storage", "pyarrow")

In [ ]:
# Montar drive:
drive.mount('/content/drive')

## Constantes


In [ ]:
# 1. Rutas originales
pf = Path('drive')
local = Path('/content')

# 2. Rutas sobre las que trabajar:
# 2.1 Rutas que apuntan hacia Drive
DRIVE_ORIGEN = pf / 'bronze'
DRIVE_DESTINO = pf / 'silver'

# 2.2 Rutas que apuntan hacia disco local:
ORIGEN_LOCAL = local / 'datos_bronze'
DESTINO_LOCAL = local / 'resultados_silver'
ORIGEN_LOCAL.mkdir(parents=True, exist_ok=True)
DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)

# 3. Escritura CSV
PARAMETROS_TO_CSV = {'sep': ";", 'index': False,
                     'encoding': "utf-8-sig", 'decimal': ","}

# 4. Limpieza y transformaciones DF
PATRONES_NULOS = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd", "x", "xx", "xxx")

## Copia archivos drive en local

In [ ]:
# Copiar todas las subcarpetas de origen drive
#pudiendo excluir algunas.

EXCLUIR = ["PRESTAMOS_HISTORICOS"]
assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

disponibles = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())
seleccion = [s for s in disponibles if s not in EXCLUIR]

faltan = [s for s in seleccion if s not in disponibles]
assert not faltan, f"No existen en DRIVE_ORIGEN: {faltan}. Disponibles: {disponibles}"

for sub in seleccion:
    shutil.copytree(DRIVE_ORIGEN / sub, ORIGEN_LOCAL / sub, dirs_exist_ok=True)

DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"Subcarpetas en Drive: {disponibles}")
print(f"Copiadas: {seleccion}\n")

for sub in seleccion:
    csvs = sorted(p.name for p in (ORIGEN_LOCAL / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

In [ ]:
# Importación archivo JSON desde Drive:

jsons = list(DRIVE_ORIGEN.glob('*.json'))
assert jsons, f'No hay ningún .json en {DRIVE_ORIGEN}'
for f in jsons:
    shutil.copy2(f, ORIGEN_LOCAL / f.name)
print(f"JSON copiados: {[f.name for f in jsons]}")


# Lectura archivo JSON para emplearlo en lectura CSV:

with open(ORIGEN_LOCAL / 'parametros_csv_brutos.json', encoding='utf-8') as f:
    params_json = json.load(f)

# Lectura CSV dimensiones prestamos, directorio bibliotecas, población, premios

### Lectura CSV - DIMENSIONES_PRESTAMOS

In [ ]:
b_dim_prestamos, fallos_dim_pre = lectura_bronze(
    params_json,
    'DIMENSIONES_PRESTAMOS',
    lista_carpetas,
    ORIGEN_CSV
    )

CSV correctamente leidos (7)
 Nombre y Dimensiones de los DF: 
codigo_lector_destino: 6 filas y 2 columnas
directorio_bibliotecas: 52 filas y 32 columnas
prcocp: 31 filas y 2 columnas
prcocs: 21 filas y 2 columnas
prcolp: 19 filas y 2 columnas
prlebi: 121 filas y 2 columnas
prprsu_prlesu: 64 filas y 3 columnas


### Lectura CSV - POBLACION-PREMIOS

In [ ]:
b_pp, fallos_pp = lectura_bronze(
    params_json,
    'POBLACION_PREMIOS',
    lista_carpetas,
    ORIGEN_CSV
    )

CSV correctamente leidos (2)
 Nombre y Dimensiones de los DF: 
300557_poblacion_distrito_barrio: 1179 filas y 10 columnas
premios_literarios_espana_2018_2026: 90 filas y 17 columnas


# Limpiezas comunes

### Limpieza DFs - DIMENSIONES_PRESTAMOS

In [ ]:
dfs_dim_prestamos, resumen_nulos_dim_p = transformaciones_comunes(b_dim_prestamos)


--------------- DF: codigo_lector_destino ---------------

1. Información incial:

1.1 Muestra:



,codigo,descripcion
5,XXX,mayores de 18 años



1.2 Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   codigo       6 non-null      string
 1   descripcion  6 non-null      string
dtypes: string(2)
memory usage: 228.0 bytes

1.3 Descripción inicial:



,codigo,descripcion
count,6,6
unique,6,6
top,ADU,adultos
freq,1,1



2. Resultados post normalizaciones del DF codigo_lector_destino:
2.1 Muestra:



,codigo,descripcion
2,juv,juvenil


... (51 outputs left out)

,biblioteca,sucursal,descripcion
62,bpm,bcdi17,bpm. biblioteca 17



2.2 Descripción:



,biblioteca,sucursal,descripcion
count,64,64,64
unique,1,64,64
top,bpm,bp02,moncloa-aravaca. bp acuna
freq,64,1,1



3. Primera inspección posibles nulos de la tabla prprsu_prlesu:

La tabla prprsu_prlesu no presenta posibles valores nulos.


### Limpieza DFs - POBLACION-PREMIOS

In [ ]:
dfs_pp, resumen_nulos_pp = transformaciones_comunes(b_pp)


--------------- DF: 300557_poblacion_distrito_barrio ---------------

1. Información incial:

1.1 Muestra:



,fecha,cod_municipio,municipio,cod_distrito,distrito,cod_barrio,barrio,num_personas,num_personas_hombres,num_personas_mujeres
427,1 de enero de 2023,28079,Madrid,6,Tetuán,64,Almenara,23.057,10.688,12.369



1.2 Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1179 entries, 0 to 1178
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   fecha                 1179 non-null   string
 1   cod_municipio         1179 non-null   string
 2   municipio             1179 non-null   string
 3   cod_distrito          1179 non-null   string
 4   distrito              1179 non-null   string
 5   cod_barrio            1179 non-null   string
 6   barrio                1179 non-null   string
 7   num_personas          1179 non-null   string
 8   num_personas_hombres  1179 non-null   string
 9   num_personas_mujeres  1179 non-null   string
dtypes: string(10)
memory usage: 175.8 KB

1.3 Descripción inicial:



,fecha,cod_municipio,municipio,cod_distrito,distrito,cod_barrio,barrio,num_personas,num_personas_hombres,num_personas_mujeres
count,1179,1179,1179,1179,1179,1179,1179,1179,1179,1179
unique,9,1,1,21,21,131,131,1158,1139,1149
top,1 de enero de 2026,28079,Madrid,15,Ciudad Lineal,11,Los Jerónimos,36.069,9.111,5.375
freq,131,1179,1179,81,81,9,10,2,3,2



2. Resultados post normalizaciones del DF 300557_poblacion_distrito_barrio:
2.1 Muestra:



,fecha,cod_municipio,municipio,cod_distrito,distrito,cod_barrio,barrio,num_personas,num_personas_hombres,num_personas_mujeres
105,1 de enero de 2026,28079,madrid,16,hortaleza,166,valdefuentes,77.038,37.592,39.446


... (8 outputs left out)


2.2 Descripción:



,premio,convocante,categoria,tipo_obra_premiada,anio_premio,anio_fallo,fecha_anuncio,fecha_verificada,rol,autor,titulo,lengua_obra,obra_en_castellano,pais_autor,autor_espanol,notas,fuente_url
count,90,90,90,90,90,90,90,90,90,90,90,90,90,90,90,90,90
unique,9,8,4,2,10,9,61,3,4,83,86,5,3,8,3,40,65
top,premio planeta,editorial planeta,novela,inedita,2024,2024,,si,ganador,"vilas, manuel",lectura facil,castellano,si,espana,si,,https://www.lecturalia.com/premio-literario/na...
freq,18,18,63,63,11,11,20,44,78,2,2,83,83,72,72,26,7



3. Primera inspección posibles nulos de la tabla premios_literarios_espana_2018_2026:


Resumen posibles nulos encontrado:



columna,autor,autor_espanol,fecha_anuncio,lengua_obra,notas,obra_en_castellano,pais_autor,titulo,total
'',1,1,20,1,26,1,1,1,52
total,1,1,20,1,26,1,1,1,52


# Limpiezas particulares por df - DIMENSIONES PRESTAMOS

## 1 - Directorio bibliotecas

In [ ]:
# Se establece variable propia para el DF:
dir_bibliotecas = dfs_dim_prestamos['directorio_bibliotecas']

### Cambio tipos de datos

Antes de proceder al cambio de tipo de dato para las columnas que les correspondan, hay que tener en cuenta que actualmente ya son de tipo String, se procede a eliminar los campos no relacionados con el objeto de la investigación.

Siendo estos:
1. horario	- apunte horario abierto al público.  
3. equipamiento - si hay wifi u otros tipos de servicio.
4. transporte - paradas metro bus cercanas al recinto.
5. descripción - Características sobre superficie del recinto.
6. accesibilidad -  Si es accesible o no.
7. content_url - web de la biblioteca
8. nombre_via
10. clase_vial
11. tipo_num
12. num
13. planta
14. puerta
15. escaleras
16. orientacion
17. coordenada_x - se deja latitud
18. coordenada_y - se deja longitud
19. telefono
20. fax
21. email
22. tipo


In [ ]:
lista_campos_ok = ['pk',
 'nombre',
 'descripcion_entidad',
 'localidad',
 'provincia',
 'codigo_postal',
 'cod_barrio',
 'barrio',
 'cod_distrito',
 'distrito',
 'latitud',
 'longitud']

In [ ]:
# Se seleccionan los campos que se quieren y se guardan en la variable:
dir_bibliotecas = dir_bibliotecas[lista_campos_ok]

In [ ]:
# Comprobación
list(dir_bibliotecas.columns)

['pk',
 'nombre',
 'descripcion_entidad',
 'localidad',
 'provincia',
 'codigo_postal',
 'cod_barrio',
 'barrio',
 'cod_distrito',
 'distrito',
 'latitud',
 'longitud']

Se modifican a tipo de dato Float los campos latitud y longitud. El resto se dejan como tipo String. Los campos de código se mantienen en string para evitar operaciones numéricas con ellos.

Al comparar valores en la búsqueda de duplicados no hay riesgo de presencia de valores nulos distintos, que duplicated los contempla como distintos.




In [ ]:
cols = ["latitud", "longitud"]
dir_bibliotecas[cols] = dir_bibliotecas[cols].apply(
    pd.to_numeric, errors="coerce").astype("float64")
#Si no se especifica tipo float, la conversión con pd.to_numeric fija tipo pandas
# Float, ya que viene de un tipo de dato pandas (string).
#revisión tipos de dato
print(dir_bibliotecas.dtypes)

pk                     string[python]
nombre                 string[python]
descripcion_entidad    string[python]
localidad              string[python]
provincia              string[python]
codigo_postal          string[python]
cod_barrio             string[python]
barrio                 string[python]
cod_distrito           string[python]
distrito               string[python]
latitud                       float64
longitud                      float64
dtype: object


### Estableciendo nulos explícitos.

In [ ]:
# Se vuelve a comprobar si coincidencias de valores en los posibles nulos:

# Selección de campos string:
cols_txt = dir_bibliotecas.select_dtypes("string").columns
# Revisión valores:
print(dir_bibliotecas[cols_txt].isin(PATRONES_NULOS).sum())

pk                      0
nombre                  0
descripcion_entidad    16
localidad               0
provincia               0
codigo_postal           0
cod_barrio              2
barrio                  0
cod_distrito            2
distrito                0
dtype: int64


In [ ]:
# Se convierten los patrones encontrados a nulos explícitos:
dir_bibliotecas[cols_txt] = dir_bibliotecas[cols_txt].replace(
    PATRONES_NULOS, pd.NA
    )

print(dir_bibliotecas[cols_txt].isna().sum())# debería cuadrar con el paso 2
print(f"Total de valores nulos: {dir_bibliotecas[cols_txt].isna().sum().sum()}")

# Se miran los registros con nulos presentes
print(f"Total registros con presencia de valores nulos: {dir_bibliotecas.isna().any(axis=1).sum()}")

pk                      0
nombre                  0
descripcion_entidad    16
localidad               0
provincia               0
codigo_postal           0
cod_barrio              2
barrio                  0
cod_distrito            2
distrito                0
dtype: int64
Total de valores nulos: 20
Total registros con presencia de valores nulos: 18


Se ve que hay 18 registros con algún valor nulo, 16 de estos valores se observan en el campo descripcion_entidad y dos registros con ausencia tanto en cod_barrio como en cod_distrito.

La alta presencia de valores nulos en descripcion_entidad, no preocupa. Son posibles ya que es campo de descripción libre.

Se procede a ver los dos registros con cod_barrio y cod_distrito nulos, ver si hay más registros con los mismos nombres de barrio y/o distrito y se asigna código de encontrarlo, si no se encuentra, se deja como nulo y se procederá a asignar de encontrarse código en el csv poblacion_distrito_barrio.

In [ ]:
# Se miran registros con nulos en los campos cod_barrio y cod_distrito

nulos_codbd = dir_bibliotecas[dir_bibliotecas[
    ["cod_barrio", "cod_distrito"]
    ].isna().any(axis=1)]
display(nulos_codbd)

,pk,nombre,descripcion_entidad,localidad,provincia,codigo_postal,cod_barrio,barrio,cod_distrito,distrito,latitud,longitud
24,1748,biblioteca publica municipal gerardo diego (vi...,la biblioteca de vallecas villa se abrio en 19...,madrid,madrid,28031,<NA>,casco historico vallecas,<NA>,villa de vallecas,40.377902,-3.622730
45,1747,biblioteca publica municipal vicalvaro,la biblioteca publica municipal de vicalvaro f...,madrid,madrid,28032,<NA>,casco historico vicalvaro,<NA>,vicalvaro,40.405284,-3.608628


In [ ]:
# Se mira si más registros con nombre de barrio y distrito
dir_bibliotecas["barrio"].value_counts().reindex(["casco historico vallecas",
                                     "casco historico vicalvaro"], fill_value=0)

,count
barrio,
casco historico vallecas,1
casco historico vicalvaro,1


In [ ]:
# Por barrio no hay más coincidencias miro por distrito
dir_bibliotecas["distrito"].value_counts().reindex(["villa de vallecas",
                                                    "vicalvaro"], fill_value=0)

,count
distrito,
villa de vallecas,3
vicalvaro,2


Por distrito hay 5 apariciones sobre las 2 por barrio, que corresponden con los registros a inspeccionar. Se ven registros, las coincidencias entre los valores de cod_distrito y distrito

In [ ]:
comp = dir_bibliotecas.value_counts(["distrito", "cod_distrito"], dropna=False)
comp[comp.index.get_level_values("distrito").isin(["villa de vallecas",
                                                    "vicalvaro"])]

distrito           cod_distrito
villa de vallecas  18              2
vicalvaro          19              1
                   <NA>            1
villa de vallecas  <NA>            1
Name: count, dtype: int64

In [ ]:
# Para ver todo el registro:
coincidencias = dir_bibliotecas[dir_bibliotecas["distrito"].isin(
    ["villa de vallecas", "vicalvaro"]
    )]
display(coincidencias)

,pk,nombre,descripcion_entidad,localidad,provincia,codigo_postal,cod_barrio,barrio,cod_distrito,distrito,latitud,longitud
7,181101,biblioteca publica luis martin santos (villa d...,<NA>,madrid,madrid,28031,2,santa eugenia,18,villa de vallecas,40.380920,-3.613673
14,50313490,biblioteca publica municipal benjamin palencia...,la biblioteca benjamin palencia esta situada e...,madrid,madrid,28051,3,ensanche de vallecas,18,villa de vallecas,40.366911,-3.616442
20,167128,biblioteca publica municipal francisco ayala (...,la biblioteca francisco ayala esta ubicada en ...,madrid,madrid,28032,2,valdebernardo,19,vicalvaro,40.399829,-3.621282
24,1748,biblioteca publica municipal gerardo diego (vi...,la biblioteca de vallecas villa se abrio en 19...,madrid,madrid,28031,<NA>,casco historico vallecas,<NA>,villa de vallecas,40.377902,-3.622730
45,1747,biblioteca publica municipal vicalvaro,la biblioteca publica municipal de vicalvaro f...,madrid,madrid,28032,<NA>,casco historico vicalvaro,<NA>,vicalvaro,40.405284,-3.608628


In [ ]:
#Se mira que en estas coincidencias la pk sea única:
coincidencias['pk'].unique()

<StringArray>
['181101', '50313490', '167128', '1748', '1747']
Length: 5, dtype: string

Hay 5 valores nulos en la pk, no coincide con ninguna biblioteca.

Se procede a asignar el valor del cod_distrito 18 para villa de vallecas y 19 para vicalvaro.

In [ ]:
# Para villa vallecas:
# ["villa de vallecas", "vicalvaro"]
maskvv = dir_bibliotecas["cod_distrito"].isna() & (dir_bibliotecas["distrito"] == "villa de vallecas")
dir_bibliotecas.loc[maskvv, "cod_distrito"] = "18"

In [ ]:
# Para villa vicalvaro:
maskvv = dir_bibliotecas["cod_distrito"].isna() & (dir_bibliotecas["distrito"] == "vicalvaro")
dir_bibliotecas.loc[maskvv, "cod_distrito"] = "19"

Quedará pendiente de asignar barrio. No necesario para el análisis.

### Mirar duplicados

In [ ]:
# PARA VER DUPLICADOS EXACTOS
print(f"Se han encontrado {dir_bibliotecas.duplicated().sum()} registros completamente duplicados")

Se han encontrado 0 registros completamente duplicados


No hay duplicados exactos. La tabla se une con tabla prprsu pero se exporta como CSV para si necsidad de revisión.

In [ ]:
dir_bibliotecas.to_csv(DRIVE_DESTINO / "directorio_bibliotecas.csv",
                       **PARAMETROS_TO_CSV)

## 2 - codigo_lector_destino

Tabla en la que se describen los códigos de catalogación del público objetivo al que está destinado el volumen. (Adulto, joven, infantil...).

En la limpieza inicial no se han detectado valores duplicados en ningún campo. Hay códigos únicos y descripciones únicas.

Se ha detectado un posible nulo en el valor "xxx" del campo codigo. Corresponde a la categoría mayores de 18 años.

### Comprobación valor nulo

In [ ]:
valor_nulo = dfs_dim_prestamos['codigo_lector_destino']['codigo'] == 'xxx'
dfs_dim_prestamos['codigo_lector_destino'][valor_nulo]

,codigo,descripcion
5,xxx,mayores de 18 anos


In [ ]:
dfs_dim_prestamos['codigo_lector_destino'] = dfs_dim_prestamos['codigo_lector_destino'].rename(columns={'codigo':'id_publico'})

### Exportar parquet
Se da por buena la tabla codigo_lector_destino. Se procede a exportar, bajo el mismo nombre que en campo de la tabla prestamos históricos.

In [ ]:
dfs_dim_prestamos['codigo_lector_destino'] .to_parquet(
    DRIVE_DESTINO / 'prcoc1.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

## 3 - prcocp: Tipo material prestado.

Tabla con los distintos códigos referenciando a los diferentes tipos de materiales disponibles en la biblioteca (libros prestables, ebook, CD...).

En la limpieza inicial se ha detectado una descripción repetida, referente a videojuegos, no es objeto del análisis así que no se trata pero dejo visualización de los registros.

No se han encontrado posibles valores nulos.

In [ ]:
dfs_dim_prestamos['prcocp'][dfs_dim_prestamos['prcocp']['descripcion'] == "juegos de consola"]

,codigo,descripcion
6,jco,juegos de consola
25,juco,juegos de consola


In [ ]:
dfs_dim_prestamos['prcocp'] = dfs_dim_prestamos['prcocp'].rename(columns={'codigo':'id_material'})

### Exportar parquet
Se da por buena la tabla prcocp. Se procede a exportar en parquet, bajo el mismo nombre, igual que en campo de la tabla prestamos históricos.

In [ ]:
dfs_dim_prestamos['prcocp'].to_parquet(
    DRIVE_DESTINO / 'prcocp.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

## 4 - prcocs: Tipo soporte ejemplar prestado

Tabla con los distintos códigos referenciando a los diferentes tipos de soporte disponibles para préstamos en la biblioteca (libros, DVDs, CD...).

En la limpieza inicial no se ha detectado ningún valor no único ni posibles valores nulos.


### Exportar Parquet
Se da por buena la tabla prcocs. Se procede a exportar, bajo el mismo nombre, igual que en campo de la tabla prestamos históricos.

Finalmente, tabla no empleada en análisis, solo se deja en tabla préstamos, los préstamos con código lib.

In [ ]:
dfs_dim_prestamos['prcocs'].to_parquet(
    DRIVE_DESTINO / 'prcocs.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

## 5 - prcolp: Tipo de lector

Tabla con los distintos códigos referenciando a los diferentes tipos lector categorizados por la biblioteca (profesor, familia, lectora...).

En la limpieza inicial no se ha detectado ningún valor no único ni posibles valores nulos.


In [ ]:
dfs_dim_prestamos['prcolp'] = dfs_dim_prestamos['prcolp'].rename(columns={'codigo':'id_lector'})

### Exportar Parquet
Se da por buena la tabla prcolp. Se procede a exportar, bajo el mismo nombre, igual que en campo de la tabla prestamos históricos.

In [ ]:
dfs_dim_prestamos['prcolp'].to_parquet(
    DRIVE_DESTINO / 'prcolp.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

## 6 - prlebi: Código red de bibliotecas

Identifica la red de bibliotecas donde el usuario se hizo el carnet de la biblioteca. Distingue entre las bibliotecas municipales y la Biblioteca Pública de Madrid (sin diferenciar estas).

En la limpieza inicial no se ha detectado ningún valor no único ni posibles valores nulos.


In [ ]:
dfs_dim_prestamos['prlebi'] = dfs_dim_prestamos['prlebi'].rename(columns={'biblioteca':'id_red'})

### Exportar Parquet
Se da por buena la tabla prlebi. Se procede a exportar, bajo el mismo nombre, igual que en campo de la tabla prestamos históricos.

In [ ]:
dfs_dim_prestamos['prlebi'].to_parquet(
    DRIVE_DESTINO / 'prlebi.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

## 7 - prprsu_prlesu: Código red de Bibliotecas Públicas de Madrid

Identifica cada biblioteca de la red de bibliotecas públicas de Madrid. Este mismo código es empleado en dos campos distintos.
- prprsu: Biblioteca donde se ha realizado el préstamo.
- prlesu: Biblioteca donde el usuario se hizo el carnet de la biblioteca.


In [ ]:
nombre = "prprsu_prlesu"
prprsu = dfs_dim_prestamos['prprsu_prlesu']

En la limpieza inicial no se ha detectado ningún valor no único ni posibles valores nulos.

Sin embargo, se observan tres campos distintos: biblioteca, sucursal y descripción. Mostrando el código de la red de bibliotecas en el campo de bibliotecas, el código de la biblioteca pública en sucursal y la descripción, compuesta por el nombre del distrito donde se ubica la biblioteca y el nombre de la biblioteca.

Se procede a eliminar el campo biblioteca ya que solo presenta un único valor para todos los registros, siendo este BPM (Biblioteca Pública de Madrid).

Así mismo, se explotará el campo descripción para que quede por un lado el nombre del distrito y por otro el nombre de la biblioteca, ambos están separados por un punto.

In [ ]:
# Eliminar campo biblioteca
prprsu = prprsu.drop(columns=['biblioteca'])

In [ ]:
# Explotar campo descripción en dos.
prprsu[["distrito", "nombre_biblioteca"]] = prprsu["descripcion"].str.split(
    ".", n=1, expand=True, regex=False
    )

# Se quitan posibles espacios:
prprsu['distrito'] = prprsu['distrito'].str.strip()
prprsu['nombre_biblioteca'] = prprsu['nombre_biblioteca'].str.strip()

In [ ]:
#Se comprueban cambios y se mira si valores únicos
prprsu.describe(include="all")

,sucursal,descripcion,distrito,nombre_biblioteca
count,64,64,64,61
unique,64,64,27,60
top,bp02,moncloa-aravaca. bp acuna,bpm,bp jose hierro
freq,1,1,7,2


In [ ]:
prprsu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   sucursal           64 non-null     string
 1   descripcion        64 non-null     string
 2   distrito           64 non-null     string
 3   nombre_biblioteca  61 non-null     string
dtypes: string(4)
memory usage: 2.1 KB


Se observan dos valores repetidos en nombre biblioteca y 7 en distrito. Se revisan registros.

In [ ]:
# Se mira registros con mismo nombre biblioteca
prprsu[prprsu.duplicated(subset="nombre_biblioteca", keep=False)]

,sucursal,descripcion,distrito,nombre_biblioteca
12,bp18,usera. bp jose hierro,usera,bp jose hierro
16,bp01,unidad central,unidad central,<NA>
36,bcdicl,bpm club de lectura,bpm club de lectura,<NA>
57,bcdiw,san blas. bp jose hierro,san blas,bp jose hierro
59,bpm,bibliotecas publicas municipales de madrid,bibliotecas publicas municipales de madrid,<NA>


Se observa que efectivamente, hay dos bibliotecas con mismo nombre pero códigos y distrito distintos. Se deja como está.

Al revisar duplicidad se ve que hay valores nulos en nombre biblioteca, ya que la descripción de la misma no pertenece a una biblioteca física sino a la descripción de una entidad superior (unidad central, bpm club de lectura y bibliotecas públicas municipales de Madrid). Se asigna valor, tanto en distrito como nombre_biblioteca "mirar descripción".

In [ ]:
# Se crea máscara booleana para filtrar por ella.

no_bibliotecas = prprsu['nombre_biblioteca'].isna()
print(f"Valores nulos en campo - nombre_biblioteca: {no_bibliotecas.sum()}")

Valores nulos en campo - nombre_biblioteca: 3


In [ ]:
# Se cambia valor descripción:
prprsu.loc[no_bibliotecas,'distrito'] = "mirar descripción"

In [ ]:
# Se cambia valor nombre_biblioteca
prprsu.loc[no_bibliotecas,'nombre_biblioteca'] = "mirar descripción"

In [ ]:
# Se revisan cambios
prprsu[no_bibliotecas]

,sucursal,descripcion,distrito,nombre_biblioteca
16,bp01,unidad central,mirar descripción,mirar descripción
36,bcdicl,bpm club de lectura,mirar descripción,mirar descripción
59,bpm,bibliotecas publicas municipales de madrid,mirar descripción,mirar descripción


In [ ]:
# Se revisa ausencia de nulos
print(f"Cantidad nulos finales: {prprsu.isna().sum().sum()}")

Cantidad nulos finales: 0


In [ ]:
# Se revisan dtypes
prprsu.dtypes

,0
sucursal,string[python]
descripcion,string[python]
distrito,string[python]
nombre_biblioteca,string[python]


### Unión con tabla bibliotecas

Se une tabla bibliotecas mediante el código de biblioteca presente en la tabla (pk).

Para ello se observa la composición de los nombres de las bibliotecas en la tabla biblioteca. Si es la misma que en la tabla prprsu, se revisa que los datos sean los mismos y se tenga registro de la misma biblioteca en ambas.

In [ ]:
# Se miran registros del dir_bibliotecas:
dir_bibliotecas['nombre'].value_counts().head(5)

,count
nombre,
biblioteca publica antonio mingote (latina),1
biblioteca publica elena fortun (retiro),1
biblioteca publica hortaleza,1
biblioteca publica javier marias (moratalaz),1
biblioteca publica jose acuna (moncloa - aravaca),1


In [ ]:
# Se miran registros de prprsu:
prprsu['nombre_biblioteca'].value_counts().head(5)

,count
nombre_biblioteca,
mirar descripción,3
bp jose hierro,2
bp luis rosales,1
bp pedro salinas,1
bp jose luis sampedro,1


Se observa que el nombre de la biblioteca en el df bibliotecas es completo. Señalando la red de biblioteca a la que pertenece, nombre de la biblioteca y distrito entre paréntesis. Para poder unificar ambas tablas, se tiene que extraer exclusivamente el nombre de la biblioteca en ambas.

Se procede a eliminar el sufijo de la tabla prprsu para dejar solo nombre. Se revisa que sea así con todas menos las ya contempladas como no biblioteca.

Por otro lado, en la tabla bibliotecas se copia campo nombre, se elimina la mención a red biblioteca y distrito.

**prprsu**

In [ ]:
# prprsu

print(f"Registros totales en prprsu: {len(prprsu)}")

#registros a modificar:
mascara = prprsu["nombre_biblioteca"].str.match(r"^\s*bp\s", case=False, na=False)
print(f"Registros con nombre que inicia por bp: {mascara.sum() }")

# Registros que nombre_biblioteca no aparece en máscara
no_mascara = ~mascara
prprsu.loc[no_mascara, "nombre_biblioteca"].value_counts(dropna=False)

Registros totales en prprsu: 64
Registros con nombre que inicia por bp: 51


,count
nombre_biblioteca,
mirar descripción,3
bpe manuel alvar,1
biblioteca 16,1
biblioteca 12,1
biblioteca 13,1
pp.,1
central,1
biblioteca 14,1
biblioteca 15,1


51 de las 64 bibliotecas contempladas inician por el prefijo bp y se podrán limpiar correctamente.

Se mira registro completo cuyo nombre inicia por bpe, para ver si anotación. También se busca por nombre específico en dir_bibliotecas.

El resto de los registros presenta nombre genérico, se miran pero no creo que se puedan determinar.

In [ ]:
# Biblioteca bpe manuel alvar

# Se mira en prprsu:
print("En prprsu:\n")
display(prprsu[prprsu['nombre_biblioteca'] == 'bpe manuel alvar'])

# Se mira en dir_bibliotecas:
print("En dir_bibliotecas:\n")
m_alvar = dir_bibliotecas['nombre'].str.contains("manuel alvar", case=False, na=False, regex=False)
display(dir_bibliotecas.loc[m_alvar, 'nombre'])

En prprsu:



,sucursal,descripcion,distrito,nombre_biblioteca
7,bp11,salamanca. bpe manuel alvar,salamanca,bpe manuel alvar


En dir_bibliotecas:



,nombre
50,biblioteca publica del estado manuel alvar (sa...


Se ve, en la tabla de dir_bibliotecas, que la biblioteca con código bp11 en la tabla prprsu es una Biblioteca Pública del Estado. Como está contemplada dentro
de prprsu, se añade a los registros de los que extraer el prefijo bp o bpe.

In [ ]:
# Se añade a la máscara la biblioteca bpe:
prprsu_alvar = prprsu["nombre_biblioteca"].str.match(r"^\s*bpe\s", case=False, na=False)

# Se miran resto de registros con nombre genérico:
no_mascara = ~(mascara | prprsu_alvar)
prprsu.loc[no_mascara]

,sucursal,descripcion,distrito,nombre_biblioteca
16,bp01,unidad central,mirar descripción,mirar descripción
17,bcdi16,bpm. biblioteca 16,bpm,biblioteca 16
18,bcdi12,bpm. biblioteca 12,bpm,biblioteca 12
19,bcdi13,bpm. biblioteca 13,bpm,biblioteca 13
24,bcdepo,bpm deposito general bb.pp.,bpm deposito general bb,pp.
36,bcdicl,bpm club de lectura,mirar descripción,mirar descripción
38,bcdid6,bpm despachos u. central,bpm despachos u,central
59,bpm,bibliotecas publicas municipales de madrid,mirar descripción,mirar descripción
60,bcdi14,bpm. biblioteca 14,bpm,biblioteca 14
61,bcdi15,bpm. biblioteca 15,bpm,biblioteca 15


Efectivamente, el resto de códigos son descriptivos de administración o no tienen nombre asignado. Se descartan para añadir información de dir_bibliotecas.

**Se procede a retirar prefijos**

In [ ]:
# Se retira prefijo bp:
prprsu['nombre_biblioteca'] = prprsu['nombre_biblioteca'].str.replace(
    r"^\s*bp\s+", "", case=False, regex=True)

# Se retira prefijo bpe:
prprsu['nombre_biblioteca'] = prprsu['nombre_biblioteca'].str.replace(
    r"^\s*bpe\s+", "", case=False, regex=True)

# Se comprueba extracción:
print("Registros en cuyo nombre se encuentra bp:")
display(prprsu[prprsu['nombre_biblioteca'].str.contains("bp", case=False, na=False, regex=False)])
# Registros sin bp o bpe:
nombres_limpios = prprsu.loc[~no_mascara]
print(f"Registros en los que no se encuentra: {len(nombres_limpios)}")
display(nombres_limpios.value_counts(dropna=False).head(2))


Registros en cuyo nombre se encuentra bp:


,sucursal,descripcion,distrito,nombre_biblioteca


Registros en los que no se encuentra: 52


,,,,count
sucursal,descripcion,distrito,nombre_biblioteca,
bcdi10,carabanchel. bp victoria prego,carabanchel,victoria prego,1
bcdi11,villa de vallecas. bp benjamin palencia,villa de vallecas,benjamin palencia,1


Ya tenemos los 52 nombres hábiles de bibliotecas limpios

In [ ]:
# Confirmo tipo dato
display(prprsu.dtypes)

# Confirmo eliminación de espacios al inicio y al final
prprsu['nombre_biblioteca'] = prprsu['nombre_biblioteca'].str.strip()
prprsu['distrito'] = prprsu['distrito'].str.strip()
prprsu['descripcion'] = prprsu['descripcion'].str.strip()

,0
sucursal,string[python]
descripcion,string[python]
distrito,string[python]
nombre_biblioteca,string[python]


**dir_bibliotecas**

In [ ]:
# Se crea nuevo campo para extraer de allí el nombre:
dir_bibliotecas['nombre_biblioteca'] = dir_bibliotecas['nombre']

print(f"Registros totales dir_biblioteca: {len(dir_bibliotecas)}."
f"\nRegistros con nombre en el nuevo campo: {dir_bibliotecas['nombre_biblioteca'].value_counts().sum()}."
f"\nRegistros nulos en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].isna().sum()}."
f"\nRegistros con valor unico en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].nunique()}.")


Regitros toales dir_biblioteca: 52.
Registros con nombre en el nuevo campo: 52.
Registros nulos en nuevo campo: 0.
Registros con valor unico en nuevo campo: 52.


In [ ]:
# Se buscan los nombres de biblioteca encontrados en prprsu
# En la tabla dir_bibliotecas y se extraen.
# 1. Valores a buscar: únicos, sin nulos ni vacíos, normalizados
valores = (prprsu["nombre_biblioteca"].dropna().str.strip().str.lower()
           .loc[lambda s: s.ne("")].drop_duplicates())

# 2. Más largos primero
valores = sorted(valores, key=len, reverse=True)

# 3. Patrón con límites de palabra
patron = r"(?<!\w)(" + "|".join(map(re.escape, valores)) + r")(?!\w)"

# 4. Extraer la primera coincidencia
dir_bibliotecas["nombre_biblioteca"] = dir_bibliotecas[
    "nombre_biblioteca"
    ].str.strip().str.lower().str.extract(patron, expand=False)

In [ ]:
# Revisión:

print(f"Registros totales dir_biblioteca: {len(dir_bibliotecas)}."
f"\nRegistros con nombre en el nuevo campo: {dir_bibliotecas['nombre_biblioteca'].value_counts().sum()}."
f"\nRegistros nulos en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].isna().sum()}."
f"\nRegistros con valor unico en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].nunique()}.")


Regitros toales dir_biblioteca: 52.
Registros con nombre en el nuevo campo: 52.
Registros nulos en nuevo campo: 0.
Registros con valor unico en nuevo campo: 50.


In [ ]:
# Vemos qué valores están duplicados
dir_bibliotecas[dir_bibliotecas.duplicated(subset="nombre_biblioteca", keep=False)]


,pk,nombre,descripcion_entidad,localidad,provincia,codigo_postal,cod_barrio,barrio,cod_distrito,distrito,latitud,longitud,nombre_biblioteca
5,123303,biblioteca publica jose hierro (usera),<NA>,madrid,madrid,28026,7,pradolongo,12,usera,40.381882,-3.710616,jose hierro
28,115056,biblioteca publica municipal jose hierro (san ...,la biblioteca publica municipal jose hierro fu...,madrid,madrid,28022,5,rosas,20,san blas-canillejas,40.433292,-3.610566,jose hierro
39,1792,biblioteca publica municipal pozo del tio raim...,la biblioteca publica pozo del tio raimundo fu...,madrid,madrid,28053,1,entrevias,13,puente de vallecas,40.373586,-3.660255,vallecas
43,1761,biblioteca publica municipal vallecas (puente ...,el 13 de noviembre de 1933 se inauguro la bibl...,madrid,madrid,28053,2,san diego,13,puente de vallecas,40.395024,-3.665554,vallecas


Hay dos grupos de duplicados, uno pertenece a las bibliotecas con nombre jose hierro una para el distrito de usera y el otro para san blas-canillejas, tal y como se ha visto en la tabla prprsu.

La otra se debe a una incorrecta extracción del nombre, del cual se ha tomado solo el nombre vallecas sin tener en cuenta el resto del mismo (pozo del tio raimundo). Se revisan

In [ ]:
# Bibliotecas jose hierro en prprsu:
prprsu[prprsu['nombre_biblioteca'] == 'jose hierro']

,sucursal,descripcion,distrito,nombre_biblioteca
12,bp18,usera. bp jose hierro,usera,jose hierro
57,bcdiw,san blas. bp jose hierro,san blas,jose hierro


El nombre del distrito no coincide en ambas tablas. Se retira -canillejas de dir_bibliotecas

In [ ]:
# distrito san blas:
san_blas = dir_bibliotecas['distrito'] == 'san blas-canillejas'
dir_bibliotecas.loc[san_blas, 'distrito'] = 'san blas'
dir_bibliotecas.loc[san_blas, 'distrito']

,distrito


In [ ]:
# Se mira si tio raimundo en prprsu
raimundo = prprsu["descripcion"].str.contains("raimundo")
prprsu.loc[raimundo]

,sucursal,descripcion,distrito,nombre_biblioteca
51,bcdiq,puente de vallecas. bp pozo tio raimundo,puente de vallecas,pozo tio raimundo


Sí está, se asigna nombre en dir_biblioteca

In [ ]:
# Asignar nombre tio raimundo en dir_bibliotecas:
raimundo_dir = dir_bibliotecas['pk'] == '1792'
dir_bibliotecas.loc[raimundo_dir, 'nombre_biblioteca'] = 'pozo tio raimundo'

In [ ]:
# Revisión
print(f"Registros totales dir_biblioteca: {len(dir_bibliotecas)}."
f"\nRegistros con nombre en el nuevo campo: "
f"{dir_bibliotecas['nombre_biblioteca'].value_counts().sum()}"
f"\nRegistros nulos en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].isna().sum()}."
f"\nRegistros con valor unico en nuevo campo: {dir_bibliotecas['nombre_biblioteca'].nunique()}."
"\nRegistros con valor unico por clave distrito-nombre_biblioteca: "
f"{dir_bibliotecas.value_counts(subset=['distrito', 'nombre_biblioteca']).sum()}")

Regitros toales dir_biblioteca: 52.
Registros con nombre en el nuevo campo: 52
Registros nulos en nuevo campo: 0.
Registros con valor unico en nuevo campo: 51.
Registros con valor unico por clave distrito-nombre_biblioteca: 52


In [ ]:
# Se comparan distrito
en_b = prprsu["distrito"].isin(dir_bibliotecas["distrito"])
prprsu[~en_b]                                   # valores de A que no están en B
print("Los valores de distrito de prprsu que no están en dir_bibliotecas: "
f"{set(prprsu["distrito"]) - set(dir_bibliotecas["distrito"])}")     # únicos de A que faltan en B

print("Los valores de distrito de dir_bibliotecas que no están en prprsu: "
f"{set(dir_bibliotecas["distrito"]) - set(prprsu["distrito"])}")          # y al revés

Los valores de distrito de prprsu que no están en dir_bibliotecas: {'bpm', 'mirar descripción', 'bpm despachos u', 'bpm deposito general bb'}
Los valores de distrito de dir_bibliotecas que no estan en prprsu: set()


Todos los distritos presentes en prprsu están en dir_bibliotecas. No hay ninguno de este último que no esté en prprsu.

In [ ]:
# Se comparan nom_biblioteca
en_b = prprsu["nombre_biblioteca"].isin(dir_bibliotecas["nombre_biblioteca"])
prprsu[~en_b]
print("Los valores de nombre_biblioteca de prprsu que no están en dir_bibliotecas: "
f"{set(prprsu["nombre_biblioteca"]) - set(dir_bibliotecas["nombre_biblioteca"])}")

print("Los valores de nombre_biblioteca de dir_bibliotecas que no están en prprsu: "
f"{set(dir_bibliotecas["nombre_biblioteca"]) - set(prprsu["nombre_biblioteca"])}")

Los valores de nombre_biblioteca de prprsu que no están en dir_bibliotecas: {'biblioteca 12', 'central', 'biblioteca 17', 'biblioteca 18', 'pp.', 'biblioteca 14', 'biblioteca 16', 'mirar descripción', 'biblioteca 13', 'biblioteca 15'}
Los valores de nombre_biblioteca de dir_bibliotecas que no estan en prprsu: set()


Todos los nombres de bibliotecas no genéricos de prprsu están en dir_bibliotecas y no falta ninguno.

In [ ]:
# Se comparan por clave ["distrito", nombre_biblioteca]
# Se comparan nom_biblioteca
en_b = prprsu[["distrito","nombre_biblioteca"]].isin(dir_bibliotecas[["distrito","nombre_biblioteca"]])
prprsu[~en_b]
print("Los valores de la clave de prprsu que no están en dir_bibliotecas: "
f"{set(prprsu[["distrito","nombre_biblioteca"]]) - set(dir_bibliotecas[["distrito","nombre_biblioteca"]])}")

print("Los valores de la clave de dir_bibliotecas que no están en prprsu: "
f"{set(dir_bibliotecas[["distrito","nombre_biblioteca"]]) - set(prprsu[["distrito","nombre_biblioteca"]])}")

Los valores de la clave de prprsu que no están en dir_bibliotecas: set()
Los valores de la clave de dir_bibliotecas que no estan en prprsu: set()


Tablas listas para utilizarse unidas mediante clave - nombre distrito.

Se crea nueva tabla donde se mantiene código prprsu.

In [ ]:
# Se crea tabla dimensión final:
t_bibliotecas = prprsu.merge(
    dir_bibliotecas[["distrito", "nombre_biblioteca","pk","cod_distrito"]],
    on=["distrito", "nombre_biblioteca"],
    how="left",
    validate="many_to_one",
    indicator=True
    )


In [ ]:
print(len(prprsu))
print(len(dir_bibliotecas))
print(len(t_bibliotecas))

64
52
64


Se miran valores nulos, únicos y dtypes:

In [ ]:
display(t_bibliotecas.isna().sum(), t_bibliotecas.nunique(), t_bibliotecas.dtypes)

,0
sucursal,0
descripcion,0
distrito,0
nombre_biblioteca,0
pk,0
cod_distrito,0
_merge,0


,0
sucursal,64
descripcion,64
distrito,22
nombre_biblioteca,63
pk,53
cod_distrito,22
_merge,2


,0
sucursal,string[pyarrow]
descripcion,string[pyarrow]
distrito,string[pyarrow]
nombre_biblioteca,string[pyarrow]
pk,string[pyarrow]
cod_distrito,string[pyarrow]
_merge,string[pyarrow]


In [ ]:
mascara = t_bibliotecas['_merge'] == 'left_only'
display(t_bibliotecas[mascara])

,sucursal,descripcion,distrito,nombre_biblioteca,pk,cod_distrito,_merge
16,bp01,unidad central,sin_referencia,unidad central,sin_referencia,sin_referencia,left_only
17,bcdi16,bpm. biblioteca 16,sin_referencia,bpm. biblioteca 16,sin_referencia,sin_referencia,left_only
18,bcdi12,bpm. biblioteca 12,sin_referencia,bpm. biblioteca 12,sin_referencia,sin_referencia,left_only
19,bcdi13,bpm. biblioteca 13,sin_referencia,bpm. biblioteca 13,sin_referencia,sin_referencia,left_only
24,bcdepo,bpm deposito general bb.pp.,sin_referencia,bpm deposito general bb.pp.,sin_referencia,sin_referencia,left_only
36,bcdicl,bpm club de lectura,sin_referencia,bpm club de lectura,sin_referencia,sin_referencia,left_only
38,bcdid6,bpm despachos u. central,sin_referencia,bpm despachos u. central,sin_referencia,sin_referencia,left_only
59,bpm,bibliotecas publicas municipales de madrid,sin_referencia,bibliotecas publicas municipales de madrid,sin_referencia,sin_referencia,left_only
60,bcdi14,bpm. biblioteca 14,sin_referencia,bpm. biblioteca 14,sin_referencia,sin_referencia,left_only
61,bcdi15,bpm. biblioteca 15,sin_referencia,bpm. biblioteca 15,sin_referencia,sin_referencia,left_only


In [ ]:
# Cambiar valores distrito, nombre_biblioteca, pk, cod_distrito para los ids
# que son códigos administrativos

t_bibliotecas.loc[mascara, 'distrito'] = 'sin_referencia'
t_bibliotecas.loc[mascara, 'nombre_biblioteca'] = t_bibliotecas['descripcion']
t_bibliotecas.loc[mascara, 'pk'] = 'sin_referencia'
t_bibliotecas.loc[mascara, 'cod_distrito'] = 'sin_referencia'

In [ ]:
t_bibliotecas.isna().sum()
t_bibliotecas['distrito'].value_counts()

,count
distrito,
sin_referencia,12
puente de vallecas,5
carabanchel,4
centro,4
hortaleza,3
latina,3
usera,3
villa de vallecas,3
san blas,3


**Cambio nombres columnas**

In [ ]:
columnas = {
    'sucursal':'id_biblioteca',
    'descripcion':'descripcion',
    'nombre_biblioteca':'nombre_biblioteca',
    'cod_distrito':'id_distrito',
    'distrito':'distrito',
    'pk':'codigo_directorio',
    'sucursal':'id_biblioteca'
}

t_bibliotecas = t_bibliotecas[list(columnas)].rename(columns=columnas)
t_bibliotecas.columns

Index(['id_biblioteca', 'descripcion', 'nombre_biblioteca', 'id_distrito',
       'distrito', 'codigo_directorio'],
      dtype='object')

### Exportar Parquet

In [ ]:
t_bibliotecas.to_parquet(
    DRIVE_DESTINO / 'prprsu.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)

# Limpiezas particulares por df - POBLACIÓN

No se realizó tratamiento individual sobre el CSV Premios por falta de tiempo.



Tabla que muestra población censada por año, distrito y barrio de la ciudad de Madrid. No presenta valores nulos.

In [ ]:
pp_p = '300557_poblacion_distrito_barrio'
poblacion = dfs_pp[pp_p]

In [ ]:
# Se observan si registros completamente duplicados:
dupli = poblacion[poblacion.duplicated(keep=False)]
print(len(dupli))
display(dupli.sort_values(list(poblacion.columns)))

0


,fecha,cod_municipio,municipio,cod_distrito,distrito,cod_barrio,barrio,num_personas,num_personas_hombres,num_personas_mujeres


### Eliminación columnas no deseadas

In [ ]:
poblacion.columns

Index(['fecha', 'cod_municipio', 'municipio', 'cod_distrito', 'distrito',
       'cod_barrio', 'barrio', 'num_personas', 'num_personas_hombres',
       'num_personas_mujeres'],
      dtype='object')

In [ ]:
cols_eliminar = ['cod_municipio', 'municipio', 'cod_barrio', 'barrio', 'num_personas_hombres',
       'num_personas_mujeres']

poblacion = poblacion.drop(columns=cols_eliminar)

In [ ]:
poblacion.columns

Index(['fecha', 'cod_municipio', 'municipio', 'cod_distrito', 'distrito',
       'cod_barrio', 'barrio', 'num_personas', 'num_personas_hombres',
       'num_personas_mujeres'],
      dtype='object')

### Establecer tipos de dato

1. Fecha, mostrando solo el año - int
2. cod_distrito y distrito - string (actual)
3. num_personas - Int64

In [ ]:
poblacion.dtypes

,0
fecha,string[pyarrow]
cod_distrito,string[pyarrow]
distrito,string[pyarrow]
num_personas,string[pyarrow]


**Columna fecha**

In [ ]:
poblacion['fecha'].unique()

<ArrowStringArray>
['1 de enero de 2026', '1 de enero de 2025', '1 de enero de 2024',
 '1 de enero de 2023', '1 de enero de 2022', '1 de enero de 2021',
 '1 de enero de 2020', '1 de enero de 2019', '1 de enero de 2018']
Length: 9, dtype: string

In [ ]:
# Se extrae únicamente el año y se guarda como tipo int
poblacion["anio"] = poblacion["fecha"].str.extract(r"(\d{4})", expand=False).astype('Int16')


In [ ]:
poblacion = poblacion.drop(columns='fecha')

In [ ]:
print(poblacion['anio'].unique())
poblacion['anio'].isna().sum()

<IntegerArray>
[2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018]
Length: 9, dtype: Int16


np.int64(0)

Columna num_personas a Int64

In [ ]:
poblacion['num_personas'].head(2)

,num_personas
0,23.410
1,48.085


In [ ]:
# Conversión a tipo Int, quitando el punto decimal, interpretación incorrecta por parte de pandas
poblacion['num_personas'] = poblacion['num_personas'].str.replace(".", "", regex=False).astype("Int64")

**Comprobaciones**

In [ ]:
poblacion.isna().sum().sum()

np.int64(0)

In [ ]:
print(poblacion.shape)
poblacion.nunique()

(1179, 4)


,0
cod_distrito,21
distrito,21
num_personas,1158
anio,9


In [ ]:
poblacion.dtypes

,0
cod_distrito,string[pyarrow]
distrito,string[pyarrow]
num_personas,Int64
anio,Int16


### Cambio nombre columnas y orden

In [ ]:
columnas = {
    'anio':'anio',
    'cod_distrito':'id_distrito',
    'distrito':'distrito',
    'num_personas':'total_poblacion'
    }

poblacion = poblacion[list(columnas)].rename(columns=columnas)

### Exportar Parquet

In [ ]:
poblacion.to_parquet(
    DRIVE_DESTINO / 'poblacion_distrito.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)